In [1]:
# ===============================================
# BOOTCAMP ML - SÉANCE 1 : SQL + DUCKDB + OLIST
# Master 2 - SEP
# ===============================================

"""
🎯 OBJECTIFS DE L'ATELIER :
1. Maîtriser DuckDB pour l'analytique rapide
2. Explorer un dataset e-commerce réaliste (Olist)
3. Créer des features pour le Machine Learning
4. Comprendre la structure relationnelle des données

⚡ POURQUOI DUCKDB ?
- SQL standard sans serveur à installer
- Parfait pour l'exploration et le prototypage
"""

"\n🎯 OBJECTIFS DE L'ATELIER :\n1. Maîtriser DuckDB pour l'analytique rapide\n2. Explorer un dataset e-commerce réaliste (Olist)\n3. Créer des features pour le Machine Learning\n4. Comprendre la structure relationnelle des données\n\n⚡ POURQUOI DUCKDB ?\n- SQL standard sans serveur à installer\n- Parfait pour l'exploration et le prototypage\n"

In [2]:

# ===============================================
# 1. INSTALLATION ET SETUP
# ===============================================

# Installation des librairies nécessaires
# !pip install duckdb pandas matplotlib seaborn plotly -q

import pandas as pd
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Setup terminé ! DuckDB est prêt à l'emploi.")
print(f"Version DuckDB : {duckdb.__version__}")
print(f"Version Pandas : {pd.__version__}")

Setup terminé ! DuckDB est prêt à l'emploi.
Version DuckDB : 1.4.3
Version Pandas : 2.3.3


In [3]:
# ===============================================
# 📊 2. CHARGEMENT DES DONNÉES OLIST
# ===============================================

print("\n" + "="*50)
print("📁 TÉLÉCHARGEMENT DU DATASET OLIST E-COMMERCE")
print("="*50)

# URL de base du dépôt officiel Olist (miroir Kaggle)
BASE_URL = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/"

# Dictionnaire des fichiers principaux
files_urls = {
    "customers": BASE_URL + "olist_customers_dataset.csv",
    "orders": BASE_URL + "olist_orders_dataset.csv",
    "order_items": BASE_URL + "olist_order_items_dataset.csv",
    "products": BASE_URL + "olist_products_dataset.csv",
    "reviews": BASE_URL + "olist_order_reviews_dataset.csv",
    "sellers": BASE_URL + "olist_sellers_dataset.csv",
}

# ===============================================
# 🔄 Tentative de chargement des données réelles
# ===============================================

try:
    print("⬇️ Chargement des données réelles Olist...")

    df_customers = pd.read_csv(files_urls["customers"])
    df_orders = pd.read_csv(files_urls["orders"])
    df_order_items = pd.read_csv(files_urls["order_items"])
    df_products = pd.read_csv(files_urls["products"])
    df_reviews = pd.read_csv(files_urls["reviews"])

    print("✅ Données Olist chargées avec succès !")

except Exception as e:
    print("⚠️ Impossible de charger les données réelles.")
    print("👉 Bascule vers des données d'exemple (structure Olist)")
    print(f"ℹ️ Raison : {e}")

# ===============================================
# 📊 Résumé des datasets
# ===============================================

print("\n📊 Résumé des datasets")
print(f"   👥 Customers    : {len(df_customers):,}")
print(f"   📦 Orders       : {len(df_orders):,}")
print(f"   🛒 Order items  : {len(df_order_items):,}")
print(f"   📱 Products     : {len(df_products):,}")
print(f"   ⭐ Reviews      : {len(df_reviews):,}")



📁 TÉLÉCHARGEMENT DU DATASET OLIST E-COMMERCE
⬇️ Chargement des données réelles Olist...
✅ Données Olist chargées avec succès !

📊 Résumé des datasets
   👥 Customers    : 99,441
   📦 Orders       : 99,441
   🛒 Order items  : 112,650
   📱 Products     : 32,951
   ⭐ Reviews      : 99,224


In [4]:
# ===============================================
# 🗃️ 3. CRÉATION DE LA BASE DE DONNÉES DUCKDB
# ===============================================

print("\n" + "="*50)
print("🗃️ CRÉATION DE LA BASE DE DONNÉES SQL")
print("="*50)

# Création d'une connexion DuckDB en mémoire
con = duckdb.connect(database=':memory:')

# Enregistrement des DataFrames pandas comme tables SQL
con.register('customers', df_customers)
con.register('orders', df_orders)
con.register('order_items', df_order_items)
con.register('products', df_products)
con.register('reviews', df_reviews)

print("✅ Base de données créée !")
print("📋 Tables disponibles :")

# Affichage des tables et leurs tailles
tables = ['customers', 'orders', 'order_items', 'products', 'reviews']
for table in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"   📊 {table}: {count:,} lignes")


🗃️ CRÉATION DE LA BASE DE DONNÉES SQL
✅ Base de données créée !
📋 Tables disponibles :
   📊 customers: 99,441 lignes
   📊 orders: 99,441 lignes
   📊 order_items: 112,650 lignes
   📊 products: 32,951 lignes
   📊 reviews: 99,224 lignes


In [5]:
# ===============================================
# 🔍 4. EXPLORATION RAPIDE
# ===============================================

# Requête 1: Aperçu des tables
print("📋 APERÇU DE LA TABLE CUSTOMERS :")
result = con.execute("SELECT * FROM customers LIMIT 5").df()
print(result)

print("\n📋 APERÇU DE LA TABLE ORDERS :")
result = con.execute("SELECT * FROM orders LIMIT 5").df()
print(result)

# Requête 2: Stats de base
print("\n📊 STATISTIQUES RAPIDES :")
stats_query = """
SELECT
    'Orders' as table_name,
    COUNT(*) as total_rows,
    COUNT(DISTINCT customer_id) as unique_customers,
    COUNT(DISTINCT order_status) as unique_statuses
FROM orders

UNION ALL

SELECT
    'Products' as table_name,
    COUNT(*) as total_rows,
    COUNT(DISTINCT product_category_name) as unique_categories,
    NULL as unique_statuses
FROM products
"""

stats_df = con.execute(stats_query).df()
print(stats_df)


📋 APERÇU DE LA TABLE CUSTOMERS :
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  

📋 APERÇU DE LA TABLE ORDERS :
                           order_id                       customer_id  \
0  e481f51cbdc54

In [6]:
# ===============================================
# 💰 5. ANALYSE BUSINESS
# ===============================================

# Question 1: Chiffre d'affaires par région
print("🗺️ CHIFFRE D'AFFAIRES PAR ÉTAT :")
ca_region_query = """
SELECT
    c.customer_state as etat,
    COUNT(DISTINCT o.order_id) as nombre_commandes,
    COUNT(DISTINCT c.customer_id) as nombre_clients,
    ROUND(SUM(oi.price + oi.freight_value), 2) as chiffre_affaires,
    ROUND(AVG(oi.price + oi.freight_value), 2) as panier_moyen
FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY 4 DESC
LIMIT 10
"""

ca_region_df = con.execute(ca_region_query).df()
print(ca_region_df)

# Visualisation
fig = px.bar(ca_region_df, x='etat', y='chiffre_affaires',
             title='Chiffre d\'affaires par État',
             color='nombre_clients', color_continuous_scale='viridis')
fig.show()


🗺️ CHIFFRE D'AFFAIRES PAR ÉTAT :
  etat  nombre_commandes  nombre_clients  chiffre_affaires  panier_moyen
0   SP             40501           40501        5771713.93        124.26
1   RJ             12350           12350        2055636.12        145.35
2   MG             11354           11354        1819357.56        140.86
3   RS              5345            5345         861734.99        140.48
4   PR              4923            4923         781851.01        138.41
5   SC              3546            3546         595135.45        145.26
6   BA              3256            3256         591137.81        160.50
7   DF              2080            2080         346428.91        147.10
8   GO              1957            1957         334662.57        146.98
9   ES              1995            1995         317701.65        142.79


## Informations Pratiques
**Format :** Travail en équipe
**Objectif :** Maîtriser SQL pour l'analyse de données et préparer un dataset ML  
**Dataset :** Base e-commerce brésilienne (Olist)

---

## 🗂️ Structure de la Base de Données

### Tables Disponibles

```sql
-- Clients
customers
-- Commandes
orders

-- Produits commandés
order_items
-- Produits
products

-- Avis clients
order_reviews

-- Vendeurs
sellers
```

---

## 🎯 NIVEAU 1 : SQL FOUNDATIONS
*Objectif : Explorer les données et comprendre le business*

### Exercice 1.1 : Vue d'ensemble des commandes
**Question :** Combien y a-t-il de commandes par statut ?

```sql
-- Calculez le nombre de commandes par statut
-- Ajoutez le pourcentage de chaque statut
-- Triez par ordre décroissant

SELECT
    order_status,
    -- TODO: Ajouter le nombre de commandes
    -- TODO: Ajoutez le calcul du pourcentage
FROM orders
GROUP BY order_status
ORDER BY nb_commandes DESC;
```

**Résultat attendu :**
```
order_status    nb_commandes    pourcentage
delivered       96478           97.0%
shipped         1107            1.1%
...
```

---

### Exercice 1.2 : Top des catégories
**Question :** Quelles sont les 10 catégories de produits les plus vendues ?

```sql
-- Joignez products et order_items
-- Comptez le nombre de ventes par catégorie
-- Limitez aux 10 premières
```

**Indices :**
- Utilisez `JOIN` entre `products` et `order_items`
- `GROUP BY product_category_name`
- `ORDER BY` + `LIMIT`

---

### Exercice 1.3 : Géographie des ventes
**Question :** Analysez les ventes par état (customer_state)

Calculez pour chaque état :
- Nombre de commandes
- Nombre de clients uniques
- Chiffre d'affaires total (price + freight_value)

```sql
-- Tables nécessaires : customers, orders, order_items
```

---
## NIVEAU 2 : JOINS & AGRÉGATIONS
*Objectif : Maîtriser les jointures complexes et les métriques business*

### Exercice 2.1 : Panier moyen par région
**Question :** Quel est le panier moyen par région ?
---

### Exercice 2.2 : Analyse de la satisfaction client
**Question :** Score moyen des avis par catégorie de produit

Colonnes attendues :
- `product_category_name`
- `avg_score` (score moyen)
- `nb_reviews` (nombre d'avis)
- `pct_satisfied` (% d'avis >= 4 étoiles)

```sql
-- Tables : products, order_items, order_reviews
-- Astuce : Utilisez CASE WHEN pour calculer le pourcentage de satisfaction
```

---

### Exercice 2.3 : Détection des clients VIP
**Question :** Identifiez les clients VIP

Critères :
- Au moins 3 commandes
- Au moins 500 BRL dépensés

Affichez :
- `customer_unique_id`
- `total_orders` (nombre de commandes)
- `total_spent` (montant total dépensé)
- `avg_order_value` (valeur moyenne du panier)
- `first_order` (date première commande)
- `last_order` (date dernière commande)

```sql
-- Tables : customers, orders, order_items
```
---

## 💎 NIVEAU 4 : EXERCICES GUIDÉS
### 📋 EXERCICE 1 : Analyse des avis clients

**Objectif :** Comprendre la satisfaction client par catégorie de produit

```sql
-- Question 1.1 : Score moyen par catégorie
SELECT
    p.product_category_name,
    ROUND(AVG(r.review_score), 2) as avg_score,
    COUNT(*) as nb_reviews
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN order_reviews r ON oi.order_id = r.order_id
GROUP BY p.product_category_name
HAVING COUNT(*) >= 10  -- Seulement catégories avec assez de données
ORDER BY avg_score DESC;

-- Question 1.2 : Distribution des scores (1 à 5 étoiles)
-- TODO: Comptez combien d'avis par score (1, 2, 3, 4, 5)

-- Question 1.3 : Top et Flop catégories
-- TODO: Identifiez les 5 meilleures et 5 pires catégories
```

---

### 📋 EXERCICE 2 : Détection de clients à risque (Churn)
**Objectif :** Identifier les clients qui pourraient partir

**Critères d'un client à risque :**
- N'a pas commandé depuis plus de 90 jours
- A un score d'avis moyen < 3 (si disponible)
- A une valeur client (CLV) significative à perdre

```sql
WITH customer_last_order AS (
    SELECT
        c.customer_unique_id,
        MAX(o.order_purchase_timestamp) as last_order_date,
        COUNT(DISTINCT o.order_id) as total_orders,
        SUM(oi.price + oi.freight_value) as total_spent
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
),
customer_satisfaction AS (
    -- TODO: Calculez le score moyen des avis par client
    -- Indice : Joignez customers, orders, order_reviews
    SELECT
        c.customer_unique_id,
        AVG(r.review_score) as avg_review_score
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_reviews r ON o.order_id = r.order_id
    GROUP BY c.customer_unique_id
)
SELECT
    clo.customer_unique_id,
    clo.last_order_date,
    -- TODO: Calculez les jours depuis la dernière commande
    CURRENT_DATE - clo.last_order_date as days_since_last_order,
    clo.total_orders,
    ROUND(clo.total_spent, 2) as clv,
    COALESCE(ROUND(cs.avg_review_score, 2), 0) as avg_review_score
FROM customer_last_order clo
LEFT JOIN customer_satisfaction cs ON clo.customer_unique_id = cs.customer_unique_id
WHERE CURRENT_DATE - clo.last_order_date > 90  -- Plus de 90 jours
  AND clo.total_spent > 100  -- CLV significative
ORDER BY clo.total_spent DESC;
```

## 💡 Tips SQL Essentiels

### Gestion des NULL
```sql
-- Remplacer les NULL par une valeur par défaut
COALESCE(avg_score, 0) as avg_score

-- Compter uniquement les non-NULL
COUNT(review_score) -- compte seulement les valeurs non-NULL
COUNT(*) -- compte toutes les lignes
```

### Date et Temps
```sql
-- Calculer la différence en jours
CURRENT_DATE - last_order_date as days_since

-- Extraire le mois
EXTRACT(MONTH FROM order_date)

-- Tronquer à la précision mois
DATE_TRUNC('month', order_date)
```

### Performance
```sql
-- BON : Filtrer avant de joindre
SELECT ...
FROM (SELECT * FROM orders WHERE order_status = 'delivered') o
JOIN customers c ON ...

-- ❌ MAUVAIS : Joindre toutes les données puis filtrer
SELECT ...
FROM orders o
JOIN customers c ON ...
WHERE o.order_status = 'delivered'
```

---

## 🎯 Livrables Attendus

1. **Fichier SQL** : Toutes vos requêtes commentées
3. **Document Markdown** :
   - Insights découverts
   - Difficultés rencontrées
   - Décisions de feature engineering

**Prochaine session :** Feature engineering en Python avec ce dataset !
---
**Bon courage ! 🚀**